In [ ]:
import pyspark
from pyspark.sql import SparkSession

In [30]:
spark.conf.set("spark.sql.parquet.enableVectorizedReader", "false")
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [ ]:
# checking the version of spark
spark.version

In [ ]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

In [ ]:
# reading the just downloaded parquet file
df_yellow = spark.read.parquet('yellow_tripdata_2025-11.parquet')b

In [ ]:
df_yellow.show()

In [13]:
df_yellow.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [14]:
   df_yellow \
        .repartition(4) \
        .write.parquet('yellow_tripdata/2025/11')

In [15]:
!tree -h yellow_tripdata/2025/11

[4.0K]  yellow_tripdata/2025/11
├── [   0]  _SUCCESS
├── [ 24M]  part-00000-11a541c7-8397-4db2-8939-8d179c816813-c000.snappy.parquet
├── [ 24M]  part-00001-11a541c7-8397-4db2-8939-8d179c816813-c000.snappy.parquet
├── [ 24M]  part-00002-11a541c7-8397-4db2-8939-8d179c816813-c000.snappy.parquet
└── [ 24M]  part-00003-11a541c7-8397-4db2-8939-8d179c816813-c000.snappy.parquet

1 directory, 5 files


In [16]:
from pyspark.sql import types

In [31]:
# using a know schmema for the yellow columns
from pyspark.sql import types

yellow_schema = types.StructType([
    # Updated to LongType to match INT64 in Parquet
    types.StructField("VendorID", types.LongType(), True), 
    types.StructField("tpep_pickup_datetime", types.TimestampType(), True),
    types.StructField("tpep_dropoff_datetime", types.TimestampType(), True),
    types.StructField("passenger_count", types.LongType(), True),
    types.StructField("trip_distance", types.DoubleType(), True),
    types.StructField("RatecodeID", types.LongType(), True),
    types.StructField("store_and_fwd_flag", types.StringType(), True),
    types.StructField("PULocationID", types.LongType(), True),
    types.StructField("DOLocationID", types.LongType(), True),
    types.StructField("payment_type", types.LongType(), True),
    # These stay as DoubleType
    types.StructField("fare_amount", types.DoubleType(), True),
    types.StructField("extra", types.DoubleType(), True),
    types.StructField("mta_tax", types.DoubleType(), True),
    types.StructField("tip_amount", types.DoubleType(), True),
    types.StructField("tolls_amount", types.DoubleType(), True),
    types.StructField("improvement_surcharge", types.DoubleType(), True),
    types.StructField("total_amount", types.DoubleType(), True),
    types.StructField("congestion_surcharge", types.DoubleType(), True)
])

In [35]:
df = spark.read \
    .parquet('yellow_tripdata/2025/11')

In [36]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [39]:
# 2. Filter for November 15th, 2025
# We use to_date to extract just the date part from the timestamp
from pyspark.sql import functions as F

nov_15_count = df.filter(F.to_date(df.tpep_pickup_datetime) == "2025-11-15").count()

In [40]:
print(f"Number of records for November 15, 2025: {nov_15_count}")

Number of records for November 15, 2025: 162604


In [41]:
df.createOrReplaceTempView("trips")

In [42]:

spark.sql("""
    SELECT count(*) as record_count 
    FROM trips 
    WHERE CAST(tpep_pickup_datetime AS DATE) = '2025-11-15'
""").show()

+------------+
|record_count|
+------------+
|      162604|
+------------+



In [43]:
# 2. Calculate duration in hours
# Unix timestamp converts time to seconds. 
# We subtract pickup from dropoff and divide by 3600.
df_with_duration = df.withColumn(
    "trip_duration_hours", 
    (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 3600
)

# 3. Get the maximum value
max_duration = df_with_duration.select(F.max("trip_duration_hours")).collect()[0][0]

print(f"The longest trip in the dataset is {max_duration:.2f} hours.")

The longest trip in the dataset is 90.65 hours.


In [44]:
# getting the zone data
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv


--2026-03-10 20:18:15--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.170.186.41, 3.170.186.111, 3.170.186.198, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.170.186.41|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-10 20:18:15 (113 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [45]:
zones = spark.read \
    .option("header", "true") \
    .csv('taxi_zone_lookup.csv')

In [46]:
zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [48]:
df_result = df.join(zones, df.PULocationID == zones.LocationID)

In [49]:
# 1. Group by the Zone name and count pickups
# 2. Sort by count in ascending order (smallest first)
# 3. Take the top result
least_frequent_zone = df_result.groupBy("Zone") \
    .count() \
    .orderBy("count", ascending=True)

# Show the results
least_frequent_zone.show(5)

+--------------------+-----+
|                Zone|count|
+--------------------+-----+
|       Arden Heights|    1|
|Eltingville/Annad...|    1|
|Governor's Island...|    1|
|       Port Richmond|    3|
|       Rikers Island|    4|
+--------------------+-----+
only showing top 5 rows

